# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Huzaif005/FlyrankAI-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My Rule and Its Reason Codes

The baseline rule assigns a survival score using simple historical observations from the Titanic dataset.

Scoring Rules:
- Female passenger → +3 points
- First Class (Pclass = 1) → +2 points
- Age below 16 years → +1 point
- Fare greater than the dataset median → +1 point

Passengers with higher scores are ranked as having a higher estimated likelihood of survival.

Reason Codes:
- RC1 = Female Passenger
- RC2 = First-Class Passenger
- RC3 = Child Passenger
- RC4 = High Ticket Fare

In [1]:
import pandas as pd
import os

# Load dataset
df = pd.read_csv("https://web.stanford.edu/class/archive/cs/cs109/cs109.1166/stuff/titanic.csv")

# Median fare
median_fare = df["Fare"].median()

# Score calculation
df["ActionScore"] = 0

df.loc[df["Sex"] == "female", "ActionScore"] += 3
df.loc[df["Pclass"] == 1, "ActionScore"] += 2
df.loc[df["Age"] < 16, "ActionScore"] += 1
df.loc[df["Fare"] > median_fare, "ActionScore"] += 1

# Build reason codes
def reason_codes(row):
    reasons = []
    if row["Sex"] == "female":
        reasons.append("RC1")
    if row["Pclass"] == 1:
        reasons.append("RC2")
    if row["Age"] < 16:
        reasons.append("RC3")
    if row["Fare"] > median_fare:
        reasons.append("RC4")
    return ",".join(reasons)

df["ReasonCodes"] = df.apply(reason_codes, axis=1)

# Rank passengers
ranked = df.sort_values("ActionScore", ascending=False)

# Save CSV
os.makedirs("work/outputs", exist_ok=True)

ranked.to_csv("work/outputs/baseline_action_score.csv", index=False)

print("CSV Saved Successfully")
print(ranked[["Name","ActionScore","ReasonCodes"]].head())

CSV Saved Successfully
                                                Name  ActionScore  \
432                         Miss. Lucile Polk Carter            7   
686                 Miss. Georgette Alexandra Madill            7   
295                      Miss. Helen Loraine Allison            7   
867  Mrs. Richard Leonard (Sallie Monypeny) Beckwith            6   
11                           Miss. Elizabeth Bonnell            6   

         ReasonCodes  
432  RC1,RC2,RC3,RC4  
686  RC1,RC2,RC3,RC4  
295  RC1,RC2,RC3,RC4  
867      RC1,RC2,RC4  
11       RC1,RC2,RC4  


## 2. Ranked Queue

Passengers were assigned a rule-based Action Score and ranked from highest to lowest. The ranked list was saved as **baseline_action_score.csv** for later review.

## 3. Top-20 review

The top 20 ranked passengers were reviewed manually.

Each passenger includes:
- Action Score
- Reason Codes
- Confidence Note
- Possible Reason the Rule Could Be Wrong

The baseline rule is intended for decision-support only and does not guarantee actual survival outcomes.

In [2]:
top20 = ranked.head(20).copy()

top20["Action"] = "High Survival Priority"

top20["Confidence"] = "Medium"

top20["What Could Be Wrong"] = (
    "Historical rule may miss other important factors."
)

review = top20[
    [
        "Name",
        "ActionScore",
        "ReasonCodes",
        "Action",
        "Confidence",
        "What Could Be Wrong",
    ]
]

print(review)


                                                  Name  ActionScore  \
432                           Miss. Lucile Polk Carter            7   
686                   Miss. Georgette Alexandra Madill            7   
295                        Miss. Helen Loraine Allison            7   
867    Mrs. Richard Leonard (Sallie Monypeny) Beckwith            6   
11                             Miss. Elizabeth Bonnell            6   
845        Mrs. Samuel L (Edwiga Grabowska) Goldenberg            6   
410                              Miss. Daisy E Minahan            6   
849                           Miss. Mary Conover Lines            6   
852          Mrs. George Dennick (Mary Hitchcock) Wick            6   
1    Mrs. John Bradley (Florence Briggs Thayer) Cum...            6   
3          Mrs. Jacques Heath (Lily May Peel) Futrelle            6   
367                       Mme. Leontine Pauline Aubart            6   
805      Mrs. Norman Campbell (Bertha Griggs) Chambers            6   
373   

## 4. Weak Picks and Leakage Check

Some passengers may receive a high Action Score based on simple rules but still not survive because other important factors are not included.

No data leakage is present because the Action Score is calculated only from features available before the survival outcome. The target variable (Survived) is never used when computing the score.

In [3]:
print("Leakage Check")

print("Is 'Survived' used in Action Score?")

print("Survived" in ["Sex","Pclass","Age","Fare"])

print("\nTop Score Distribution")

print(ranked["ActionScore"].value_counts())


Leakage Check
Is 'Survived' used in Action Score?
False

Top Score Distribution
ActionScore
0    323
3    211
1     92
6     91
4     90
2     42
5     35
7      3
Name: count, dtype: int64


In [4]:
os.makedirs("work/outputs", exist_ok=True)

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.